# NER with CNN-BiLSTM

Giuseppe Termerissa


## Setup


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import os
import json
import numpy as np
import glob
import shutil
import tensorflow

import sys

sys.path.insert(0, "./src/")

### Download dependencies

This is unnecessary unless the notebook is being ran on Colab with no pre-downloaded datasets, and will be normally skipped.


In [ ]:
try:
    import google.colab  # type: ignore

    IN_COLAB = True
except Exception:
    IN_COLAB = False

In [ ]:
if "glove.6B" not in os.listdir("../data"):
    !wget "https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip"

    # four file with different numbers of parameters: 50, 100, 200, 300
    !unzip glove.6B.zip -d ./data/glove.6B/

In [ ]:
# download and unzip OntoNotes5.0/CoNLL-2012 dataset
if "conll-2012" not in os.listdir("./data"):
    !wget "https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/zmycy7t9h9-2.zip"
    !unzip zmycy7t9h9-2.zip -d ./data/tmp_conll2012
    !unzip data/tmp_conll2012/zmycy7t9h9-2/conll-2012.zip -d data/
    shutil.rmtree("./data/tmp_conll2012")

## Pipeline


In [ ]:
hyper = {
    # CNN hyperameters
    "CONV_WIDTH": 4,
    "FILTERS": 30,
    # BiLSTM hyperparameters
    "LSTM_STATE_SIZE": 300,
    "LSTM_DROPOUT": 0.68,
    "REC_DROPOUT": 0.25,
    # Model hyperparameters
    "WORD_EMB_DIR": "glove.6B/glove.6B.100d.txt",
    "EPOCHS": 19,
    "BATCH_SIZE": 128,
    "OPTIMIZER": "rmsprop",
}

In [ ]:
from src.datasets import load_conll2003, load_conll2012

conll2012_dir = "../data/conll-2012"
conll2003_dir = "../data/conll003-englishversion"


train_sentences, dev_sentences, test_sentences = load_conll2012(conll2012_dir)

len(train_sentences), len(test_sentences)

In [ ]:
from src.pipeline import unpack_sentences_conll2012
from src.pipeline import build_labels_lu
from src.pipeline import caps_lu
from src.pipeline import load_word_embeddings
from src.pipeline import create_char_lu

labels, vocab = unpack_sentences_conll2012([train_sentences, test_sentences])

labels_lu = build_labels_lu(labels)

case_emb = np.identity(len(caps_lu), dtype="float32")

words_lu, word_emb = load_word_embeddings(f"../data/{hyper['WORD_EMB_DIR']}", vocab)

char_lu = create_char_lu()

In [ ]:
from src.pipeline import sentences_to_matrix_conll2012

train_set = sentences_to_matrix_conll2012(
    train_sentences, caps_lu, words_lu, char_lu, labels_lu
)
test_set = sentences_to_matrix_conll2012(
    test_sentences, caps_lu, words_lu, char_lu, labels_lu
)
dev_set = sentences_to_matrix_conll2012(
    dev_sentences, caps_lu, words_lu, char_lu, labels_lu
)

## Model Setup


In [ ]:
from src.model import create_model_conll2012


model = create_model_conll2012(
    hyper,
    word_emb,
    case_emb,
    char_lu,
    labels_lu,
)

## Training


In [ ]:
from src.pipeline import pad_dataset_conll2012

tokens, caps, chars, labels = pad_dataset_conll2012(train_set)

# validation set for CoNLL-2012
dev_tokens, dev_caps, dev_chars, dev_labels = pad_dataset_conll2012(dev_set)
X_dev = [dev_tokens, dev_caps, dev_chars]
y_dev = dev_labels

X = [tokens, caps, chars]
y = labels

history = model.fit(
    X,
    y,
    epochs=hyper["EPOCHS"],
    batch_size=hyper["BATCH_SIZE"],
    validation_data=(X_dev, y_dev),
)

## Evaluation


In [ ]:
# compute predictions on the test set
from src.evaluation import predict_dataset

y_true, y_pred = predict_dataset(test_set, model)

# reverse label lookup
label_idx_lu = {v: k for k, v in labels_lu.items()}

# convert back to labels
pred_label = []
for sentence in y_pred:
    pred_label.append([label_idx_lu[element] for element in sentence])

true_label = []
for sentence in y_true:
    true_label.append([label_idx_lu[element] for element in sentence])

In [ ]:
from src.evaluation import evaluate_pred

pre, rec, f1 = evaluate_pred(pred_label, true_label)

print(f"Precision:  {pre:.3f}")
print(f"Recall:     {rec:.3f}")
print(f"F1-Score:   {f1:.3f}")